# Bird-only dataset manifest

This notebook creates a manifest containing only birds from the iNaturalist mini training dataset. Bird membership is determined from the source JSON: a category is included only when `class == "Aves"`.
## Outputs
All generated files are written to `bird_manifest/`:

- `bird_manifest.csv` — one row per bird image, including taxonomy, source metadata,
  licence, absolute/relative paths, existence, and on-disk size
- `bird_categories.csv` — one row per bird category/species
- `bird_category_summary.csv` — bird image counts by category
- `missing_bird_images.csv` — metadata records whose expected image is absent
- `bird_manifest_summary.json` — run configuration, counts, and validation results

CSV files use UTF-8 with a BOM for spreadsheet compatibility. Exact observation
coordinates are retained from the source metadata; handle them according to your
privacy and data-governance requirements.


## 1. Configuration and helpers


In [1]:
import numpy as np
print(np.__version__)

import pandas as pd
print(pd.__version__)

import sklearn
print(sklearn.__version__)

import cv2
print(cv2.__version__)

1.26.4
3.0.6
1.9.1
4.14.0


In [5]:
from __future__ import annotations

import csv
import json
import os
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from typing import Any, Iterable

try:
    import pandas as pd
except ImportError:
    pd = None


DATASET_ROOT = Path(r"E:\Datasets\iNaturalist")
SPLIT_ROOT = DATASET_ROOT / "train_mini"
IMAGE_ROOT = SPLIT_ROOT / "train_mini"

METADATA_PATH = SPLIT_ROOT / "train_mini.json" / "train_mini.json"

OUTPUT_DIR = Path(r"C:\Users\Joe\Desktop\ESP Classifier\esp32_Classifier\Notebooks")

EXPECTED_BIRD_CATEGORIES = 1_486
EXPECTED_BIRD_IMAGES = 74_300


def write_csv(path: Path, rows: Iterable[dict[str, Any]], fields: list[str]) -> int:
    count = 0
    with path.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field) for field in fields})
            count += 1
    return count


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"Metadata not found at {METADATA_PATH}. Set BIRD_DATASET_ROOT to the directory "
        "that contains train_mini."
    )
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f"Image root not found: {IMAGE_ROOT}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python:        {sys.version.split()[0]}")
print(f"Dataset root:  {DATASET_ROOT}")
print(f"Metadata:      {METADATA_PATH}")
print(f"Image root:    {IMAGE_ROOT}")
print(f"Output:        {OUTPUT_DIR}")


Python:        3.11.16
Dataset root:  E:\Datasets\iNaturalist
Metadata:      E:\Datasets\iNaturalist\train_mini\train_mini.json\train_mini.json
Image root:    E:\Datasets\iNaturalist\train_mini\train_mini
Output:        C:\Users\Joe\Desktop\ESP Classifier\esp32_Classifier\Notebooks


## 2. Load and validate the iNaturalist/COCO metadata


In [7]:
with METADATA_PATH.open("r", encoding="utf-8") as handle:
    document = json.load(handle)

required_sections = {"images", "annotations", "categories", "licenses"}
missing_sections = required_sections - document.keys()
if missing_sections:
    raise ValueError(f"Metadata is missing required sections: {sorted(missing_sections)}")

images = document["images"]
annotations = document["annotations"]
categories = document["categories"]
licenses = document["licenses"]

category_by_id = {item["id"]: item for item in categories}
image_by_id = {item["id"]: item for item in images}
license_by_id = {item["id"]: item for item in licenses}

if len(category_by_id) != len(categories):
    raise ValueError("Duplicate category IDs found")
if len(image_by_id) != len(images):
    raise ValueError("Duplicate image IDs found")

annotation_counts = Counter(item["image_id"] for item in annotations)
duplicate_annotation_image_ids = [key for key, count in annotation_counts.items() if count != 1]
unknown_image_annotations = [
    item["id"] for item in annotations if item["image_id"] not in image_by_id
]
unknown_category_annotations = [
    item["id"] for item in annotations if item["category_id"] not in category_by_id
]
unannotated_image_ids = set(image_by_id) - annotation_counts.keys()

if duplicate_annotation_image_ids:
    raise ValueError(
        f"Expected one annotation per image; {len(duplicate_annotation_image_ids)} image IDs differ"
    )
if unknown_image_annotations or unknown_category_annotations or unannotated_image_ids:
    raise ValueError(
        "Broken annotation references: "
        f"unknown images={len(unknown_image_annotations)}, "
        f"unknown categories={len(unknown_category_annotations)}, "
        f"unannotated images={len(unannotated_image_ids)}"
    )

print(f"Images:       {len(images):,}")
print(f"Annotations:  {len(annotations):,}")
print(f"Categories:   {len(categories):,}")
print(f"Licences:     {len(licenses):,}")


Images:       500,000
Annotations:  500,000
Categories:   10,000
Licences:     7


## 3. Select `Aves` records and resolve physical image paths


In [9]:
bird_categories = sorted(
    (item for item in categories if item.get("class") == "Aves"),
    key=lambda item: item["id"],
)
bird_category_ids = {item["id"] for item in bird_categories}
bird_annotations = [
    item for item in annotations if item["category_id"] in bird_category_ids
]

if len(bird_categories) != EXPECTED_BIRD_CATEGORIES:
    raise AssertionError(
        f"Expected {EXPECTED_BIRD_CATEGORIES:,} bird categories, found {len(bird_categories):,}"
    )
if len(bird_annotations) != EXPECTED_BIRD_IMAGES:
    raise AssertionError(
        f"Expected {EXPECTED_BIRD_IMAGES:,} bird images, found {len(bird_annotations):,}"
    )

manifest_rows: list[dict[str, Any]] = []
for annotation in bird_annotations:
    image = image_by_id[annotation["image_id"]]
    category = category_by_id[annotation["category_id"]]
    licence = license_by_id.get(image.get("license"), {})

    # Source JSON paths contain train_mini/<class>/<file>
    image_name = PurePosixPath(image["file_name"]).name
    absolute_path = IMAGE_ROOT / category["image_dir_name"] / image_name
    exists = absolute_path.is_file()

    manifest_rows.append({
        "image_id": image["id"],
        "annotation_id": annotation["id"],
        "category_id": category["id"],
        "relative_path": absolute_path.relative_to(DATASET_ROOT).as_posix(),
        "absolute_path": str(absolute_path),
        "source_file_name": image["file_name"],
        "file_exists": exists,
        "file_size_bytes": absolute_path.stat().st_size if exists else None,
        "width": image.get("width"),
        "height": image.get("height"),
        "scientific_name": category.get("name"),
        "common_name": category.get("common_name"),
        "supercategory": category.get("supercategory"),
        "kingdom": category.get("kingdom"),
        "phylum": category.get("phylum"),
        "class": category.get("class"),
        "order": category.get("order"),
        "family": category.get("family"),
        "genus": category.get("genus"),
        "specific_epithet": category.get("specific_epithet"),
        "image_dir_name": category.get("image_dir_name"),
        "date": image.get("date"),
        "rights_holder": image.get("rights_holder"),
        "latitude": image.get("latitude"),
        "longitude": image.get("longitude"),
        "location_uncertainty": image.get("location_uncertainty"),
        "license_id": image.get("license"),
        "license_name": licence.get("name"),
        "license_url": licence.get("url"),
    })

missing_rows = [row for row in manifest_rows if not row["file_exists"]]
print(f"Bird categories:       {len(bird_categories):,}")
print(f"Bird image records:    {len(manifest_rows):,}")
print(f"Physical files found:  {len(manifest_rows) - len(missing_rows):,}")
print(f"Physical files missing:{len(missing_rows):,}")


Bird categories:       1,486
Bird image records:    74,300
Physical files found:  74,300
Physical files missing:0


## 4. Write manifests and summaries


In [11]:
MANIFEST_FIELDS = [
    "image_id", "annotation_id", "category_id", "relative_path", "absolute_path",
    "source_file_name", "file_exists", "file_size_bytes", "width", "height",
    "scientific_name", "common_name", "supercategory", "kingdom", "phylum", "class",
    "order", "family", "genus", "specific_epithet", "image_dir_name", "date",
    "rights_holder", "latitude", "longitude", "location_uncertainty", "license_id",
    "license_name", "license_url",
]
CATEGORY_FIELDS = [
    "id", "name", "common_name", "supercategory", "kingdom", "phylum", "class",
    "order", "family", "genus", "specific_epithet", "image_dir_name",
]
MISSING_FIELDS = [
    "image_id", "annotation_id", "category_id", "scientific_name", "common_name",
    "relative_path", "absolute_path", "source_file_name",
]

manifest_count = write_csv(
    OUTPUT_DIR / "bird_manifest.csv", manifest_rows, MANIFEST_FIELDS
)
category_count = write_csv(
    OUTPUT_DIR / "bird_categories.csv", bird_categories, CATEGORY_FIELDS
)
missing_count = write_csv(
    OUTPUT_DIR / "missing_bird_images.csv", missing_rows, MISSING_FIELDS
)

category_image_counts = Counter(row["category_id"] for row in manifest_rows)
category_summary_rows = [
    {
        "category_id": item["id"],
        "scientific_name": item.get("name"),
        "common_name": item.get("common_name"),
        "order": item.get("order"),
        "family": item.get("family"),
        "image_count": category_image_counts[item["id"]],
    }
    for item in bird_categories
]
category_summary_rows.sort(
    key=lambda row: (-row["image_count"], row["scientific_name"] or "")
)
write_csv(
    OUTPUT_DIR / "bird_category_summary.csv",
    category_summary_rows,
    ["category_id", "scientific_name", "common_name", "order", "family", "image_count"],
)

image_ids = [row["image_id"] for row in manifest_rows]
annotation_ids = [row["annotation_id"] for row in manifest_rows]
all_aves = all(row["class"] == "Aves" for row in manifest_rows)
unique_image_ids = len(image_ids) == len(set(image_ids))
unique_annotation_ids = len(annotation_ids) == len(set(annotation_ids))
category_totals_match = sum(category_image_counts.values()) == manifest_count

validations = {
    "all_manifest_rows_are_aves": all_aves,
    "image_ids_are_unique": unique_image_ids,
    "annotation_ids_are_unique": unique_annotation_ids,
    "category_totals_match_manifest": category_totals_match,
    "manifest_count_matches_expected": manifest_count == EXPECTED_BIRD_IMAGES,
    "category_count_matches_expected": category_count == EXPECTED_BIRD_CATEGORIES,
}
if not all(validations.values()):
    raise AssertionError(f"Manifest validation failed: {validations}")

summary = {
    "generated_utc": utc_now(),
    "dataset_root": str(DATASET_ROOT),
    "metadata_path": str(METADATA_PATH),
    "image_root": str(IMAGE_ROOT),
    "output_directory": str(OUTPUT_DIR),
    "selection_rule": 'category["class"] == "Aves"',
    "source_counts": {
        "images": len(images),
        "annotations": len(annotations),
        "categories": len(categories),
        "licenses": len(licenses),
    },
    "bird_counts": {
        "images": manifest_count,
        "categories": category_count,
        "files_found": manifest_count - missing_count,
        "files_missing": missing_count,
    },
    "validations": validations,
    "outputs": [
        "bird_manifest.csv", "bird_categories.csv", "bird_category_summary.csv",
        "missing_bird_images.csv", "bird_manifest_summary.json",
    ],
}
with (OUTPUT_DIR / "bird_manifest_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)

print(json.dumps(summary, indent=2))


{
  "generated_utc": "2026-09-29T01:38:39.790990+00:00",
  "dataset_root": "E:\\Datasets\\iNaturalist",
  "metadata_path": "E:\\Datasets\\iNaturalist\\train_mini\\train_mini.json\\train_mini.json",
  "image_root": "E:\\Datasets\\iNaturalist\\train_mini\\train_mini",
  "output_directory": "C:\\Users\\Joe\\Desktop\\ESP Classifier\\esp32_Classifier\\Notebooks",
  "selection_rule": "category[\"class\"] == \"Aves\"",
  "source_counts": {
    "images": 500000,
    "annotations": 500000,
    "categories": 10000,
    "licenses": 7
  },
  "bird_counts": {
    "images": 74300,
    "categories": 1486,
    "files_found": 74300,
    "files_missing": 0
  },
  "validations": {
    "all_manifest_rows_are_aves": true,
    "image_ids_are_unique": true,
    "annotation_ids_are_unique": true,
    "category_totals_match_manifest": true,
    "manifest_count_matches_expected": true,
    "category_count_matches_expected": true
  },
  "outputs": [
    "bird_manifest.csv",
    "bird_categories.csv",
    "bird_c

## 5. Preview and distribution checks


In [13]:
order_counts = Counter(row["order"] for row in manifest_rows)
family_counts = Counter(row["family"] for row in manifest_rows)

print("Largest bird orders by image count:")
for name, count in order_counts.most_common(10):
    print(f"  {name or '[missing]':30} {count:>6,}")

print("\nLargest bird families by image count:")
for name, count in family_counts.most_common(10):
    print(f"  {name or '[missing]':30} {count:>6,}")

if pd is not None:
    display(pd.DataFrame(manifest_rows).head(10))
    display(pd.DataFrame(category_summary_rows).head(10))
else:
    print("\nFirst five manifest records:")
    for row in manifest_rows[:5]:
        print(row)


Largest bird orders by image count:
  Passeriformes                  33,900
  Charadriiformes                 8,200
  Anseriformes                    4,600
  Accipitriformes                 3,800
  Pelecaniformes                  3,050
  Piciformes                      2,650
  Caprimulgiformes                2,200
  Psittaciformes                  2,050
  Columbiformes                   1,800
  Coraciiformes                   1,550

Largest bird families by image count:
  Anatidae                        4,500
  Accipitridae                    3,700
  Scolopacidae                    2,650
  Laridae                         2,450
  Parulidae                       2,350
  Tyrannidae                      2,250
  Picidae                         2,150
  Passerellidae                   2,050
  Corvidae                        2,000
  Thraupidae                      2,000


,image_id,annotation_id,category_id,relative_path,absolute_path,source_file_name,file_exists,file_size_bytes,width,height,...,specific_epithet,image_dir_name,date,rights_holder,latitude,longitude,location_uncertainty,license_id,license_name,license_url
0,53,53,3360,train_mini/train_mini/03360_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/03360_Animalia_Chordata_Aves_Charad...,True,97139,429,417,...,ruficapillus,03360_Animalia_Chordata_Aves_Charadriiformes_C...,2007-10-22 00:00:00+00:00,megatherium,-30.50815,115.08134,NaN,2,Attribution-NonCommercial-NoDerivatives License,http://creativecommons.org/licenses/by-nc-nd/4.0/
1,73,73,3533,train_mini/train_mini/03533_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/03533_Animalia_Chordata_Aves_Columb...,True,119875,500,401,...,chalcoptera,03533_Animalia_Chordata_Aves_Columbiformes_Col...,2007-11-02 00:00:00+00:00,megatherium,-33.36702,115.61973,NaN,2,Attribution-NonCommercial-NoDerivatives License,http://creativecommons.org/licenses/by-nc-nd/4.0/
2,82,82,4087,train_mini/train_mini/04087_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/04087_Animalia_Chordata_Aves_Passer...,True,66929,500,372,...,townsendi,04087_Animalia_Chordata_Aves_Passeriformes_Par...,2011-10-08 00:00:00+00:00,greenrosettas,37.83727,-122.23475,NaN,1,Attribution-NonCommercial License,http://creativecommons.org/licenses/by-nc/4.0/
3,90,90,3357,train_mini/train_mini/03357_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/03357_Animalia_Chordata_Aves_Charad...,True,60711,500,350,...,nivosus,03357_Animalia_Chordata_Aves_Charadriiformes_C...,2011-12-10 00:00:00+00:00,cyric,29.60591,-85.44230,29445.0,3,Attribution-NonCommercial-ShareAlike License,http://creativecommons.org/licenses/by-nc-sa/4.0/
4,99,99,4486,train_mini/train_mini/04486_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/04486_Animalia_Chordata_Aves_Procel...,True,34897,500,480,...,immutabilis,04486_Animalia_Chordata_Aves_Procellariiformes...,2010-01-07 00:00:00+00:00,kestrel,22.31694,-159.47099,30298.0,1,Attribution-NonCommercial License,http://creativecommons.org/licenses/by-nc/4.0/
5,145,145,3264,train_mini/train_mini/03264_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/03264_Animalia_Chordata_Aves_Anseri...,True,23309,500,348,...,jamaicensis,03264_Animalia_Chordata_Aves_Anseriformes_Anat...,2011-12-23 00:00:00+00:00,dloarie,38.22321,-122.59476,NaN,0,Attribution License,http://creativecommons.org/licenses/by/4.0/
6,168,168,4574,train_mini/train_mini/04574_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/04574_Animalia_Chordata_Aves_Sulifo...,True,62772,500,338,...,auritus,04574_Animalia_Chordata_Aves_Suliformes_Phalac...,2011-12-20 00:00:00+00:00,John Reeves,25.28662,-80.89865,NaN,2,Attribution-NonCommercial-NoDerivatives License,http://creativecommons.org/licenses/by-nc-nd/4.0/
7,178,178,3508,train_mini/train_mini/03508_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/03508_Animalia_Chordata_Aves_Ciconi...,True,32176,500,354,...,americana,03508_Animalia_Chordata_Aves_Ciconiiformes_Cic...,2011-12-22 00:00:00+00:00,John Reeves,25.61231,-80.72076,29936.0,2,Attribution-NonCommercial-NoDerivatives License,http://creativecommons.org/licenses/by-nc-nd/4.0/
8,213,213,4434,train_mini/train_mini/04434_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/04434_Animalia_Chordata_Aves_Picifo...,True,37172,280,347,...,major,04434_Animalia_Chordata_Aves_Piciformes_Picida...,2011-04-15 00:00:00+00:00,megatherium,52.48327,13.41720,NaN,2,Attribution-NonCommercial-NoDerivatives License,http://creativecommons.org/licenses/by-nc-nd/4.0/
9,265,265,3479,train_mini/train_mini/03479_Animalia_Chordata_...,E:\Datasets\iNaturalist\train_mini\train_mini\...,train_mini/03479_Animalia_

,category_id,scientific_name,common_name,order,family,image_count
0,3806,Acanthis cabaret,Lesser Redpoll,Passeriformes,Fringillidae,50
1,3807,Acanthis flammea,Common Redpoll,Passeriformes,Fringillidae,50
2,3808,Acanthis hornemanni,Hoary Redpoll,Passeriformes,Fringillidae,50
3,3677,Acanthisitta chloris,Rifleman,Passeriformes,Acanthisittidae,50
4,3678,Acanthiza chrysorrhoa,Yellow-rumped Thornbill,Passeriformes,Acanthizidae,50
5,3679,Acanthiza pusilla,Brown Thornbill,Passeriformes,Acanthizidae,50
6,3923,Acanthorhynchus tenuirostris,Eastern Spinebill,Passeriformes,Meliphagidae,50
7,3111,Accipiter badius,Shikra,Accipitriformes,Accipitridae,50
8,3112,Accipiter cooperii,Cooper's Hawk,Accipitriformes,Accipitridae,50
9,3113,Accipiter gentilis,Northern Goshawk,Accipitriformes,Accipitridae,50
